# Day 2 Homework: Website Summarizer with a Local Model

The day 1 summarizer, upgraded to run on an open-source model via Ollama instead of a paid API.

The only things that change are the `base_url`, the `api_key` and the `model` - the rest of the code is the same.
At the end we run the same website through Claude too, to compare a small local model with a frontier model.

In [ ]:
# imports

import os
import requests
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
# Check Ollama is running - if not, open a terminal and run `ollama serve`

OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_MODEL = "llama3.2:1b"  # change to "llama3.2" if your machine can handle the bigger model

requests.get("http://localhost:11434").content

In [ ]:
# Same OpenAI library, pointed at the local Ollama server - the api_key can be anything

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
# The same prompts as day 1

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
# summarize now takes the client and model, so we can swap providers freely

def summarize(url, client=ollama, model=OLLAMA_MODEL):
    website = fetch_website_contents(url)
    response = client.chat.completions.create(
        model = model,
        messages = messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url, client=ollama, model=OLLAMA_MODEL):
    summary = summarize(url, client, model)
    display(Markdown(summary))

In [ ]:
# Runs entirely on your machine - no API charges, and the data never leaves your box

display_summary("https://edwarddonner.com")

In [ ]:
display_summary("https://anthropic.com")

## Bonus: compare with Claude

Same function, just a different client and model. Notice the difference in quality between a 1B local model and a frontier model.

In [ ]:
load_dotenv(override=True)

claude = OpenAI(api_key=os.getenv("ANTHROPIC_API_KEY"), base_url="https://api.anthropic.com/v1/")

display_summary("https://edwarddonner.com", client=claude, model="claude-haiku-4-5")